# Lab 3 — Recursive context decomposition (EP07)

In Lab 2 every method accepted the same constraint: everything must fit in one window
§18.2.5 offers another way — **no single call sees the full context**:

**RLM(q, C) = M(q, RLM(q₁, C₁), RLM(q₂, C₂), …)** (Equation 18.7)

Decompose → recurse on each chunk → aggregate, following the textbook's `recursive_summarize`

**What you'll learn**
1. When a document is bigger than the budget: cut it to fit and send once — only the start is seen
2. Write the textbook's `recursive_summarize` and look at the tree of calls
3. A real pitfall we hit: after splitting, **the heading is lost**
4. The cost: how many calls, and how big the biggest call is

The meeting notes in the document are **simulated data generated for teaching**. Every model answer comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §18.2.5 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **3 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep07-lab3-recursive-context.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs under one US cent) |
| Packages | `openai`, `python-dotenv`, `tokenizers` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "tokenizers==0.23.3"

## 1. The model and token counting (from Lab 1)

Gemma 4 26B through OpenRouter, provider pinned to DeepInfra — why we chose the model and how to set up the key are in Lab 1
`call` calls the model at temperature 0 and records in `CALLS` which level of the tree each call is at and how many tokens it sent

In [ ]:
import os, re, random
from openai import OpenAI
os.environ["HF_HUB_VERBOSITY"] = "error"
from tokenizers import Tokenizer

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False}, "provider": {"order": ["deepinfra/fp8"], "allow_fallbacks": False}}
CALLS = []   # (depth, input tokens, output tokens) of every call

def call(prompt, depth=0):
    reply = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}],
                                           max_tokens=500, temperature=0, extra_body=EXTRA)
    CALLS.append((depth, reply.usage.prompt_tokens, reply.usage.completion_tokens))
    return reply.choices[0].message.content

tok = Tokenizer.from_pretrained(MODEL)

def count_tokens(text):
    return len(tok.encode(text, add_special_tokens=False).ids)

## 2. A document bigger than the budget

A research group's meeting notes for a whole year (52 weeks · simulated data) — decisions about papers are spread over three places,
and week 27 is a **full meeting transcript**, much longer than the other weeks

The question: *which papers did the group decide to drop from the review's main table, why, and in which week?*

This lab's budget per call: **1,500 tokens** (kept small so the tree is easy to see — the same principle as a document of hundreds of thousands of tokens)

In [ ]:
random.seed(3)  # simulated meeting notes, generated for teaching
PEOPLE = ["Nok", "Ton", "Ploy", "Bank", "Mint"]
TASKS = ["data loaders for the weather set", "the GPU booking sheet", "a reading list for new members", "plots for the seminar",
         "the baseline scripts", "the shared bibliography file", "unit tests for the metrics", "the poster draft"]
DECISIONS = {
    9:  "- Decision: drop P09 (Temporal Fusion Transformer) from the main table, because it was published in 2019, before our 2023 cutoff.",
    30: "- Ploy suggested dropping P05 (PatchTST), but we decided to keep it: it is the strongest baseline we have.",
    44: "- Decision: drop P08 (Chronos) from the main table, because its benchmark used a different forecast horizon, so the numbers are not comparable.",
}

def week(k):
    a, b = random.sample(PEOPLE, 2)
    lines = [f"## Week {k}", f"Present: {', '.join(random.sample(PEOPLE, 4))}."]
    for _ in range(5):
        lines.append(f"- {random.choice(PEOPLE)} worked on {random.choice(TASKS)}; about {random.randint(2, 9)} hours, "
                     f"{random.choice(['done', 'half done', 'blocked on data', 'needs review'])}.")
    lines.append(f"- Next week: {a} presents, {b} takes notes.")
    return "\n".join(lines)

def transcript(k):
    lines = [f"## Week {k} (full transcript of the review meeting)"]
    for i in range(60):
        lines.append(f"{random.choice(PEOPLE)}: On {random.choice(TASKS)}, I think we should spend {random.randint(1, 6)} more hours "
                     f"before the seminar, and check it with {random.choice(PEOPLE)} first.")
        if i % 6 == 5:
            lines.append("")
        if i == 33:
            lines.append("Nok: So we agree: drop P04 (Are Transformers Effective for Time Series Forecasting?) from the main table, "
                         "because it is a critique of transformers, not a new model. It moves to the discussion section.")
    return "\n".join(lines)

weeks = []
for k in range(1, 53):
    w = transcript(k) if k == 27 else week(k)
    weeks.append(w + ("\n" + DECISIONS[k] if k in DECISIONS else ""))
DOC = "\n\n".join(["# Cogito Learn research group: meeting notes, 2026"] + weeks)

In [ ]:
MAX = 1500
QUERY = ("Which papers did we decide to drop from the review's main table, and why? "
         "Answer one line per paper: paper id, week, reason.")
print(f"document: {count_tokens(DOC):,} tokens · budget per call: {MAX:,}")

def report(answer):
    print(answer)
    found = {}
    for p in ["P09", "P04", "P08"]:
        lines = [l for l in answer.splitlines() if p in l]
        if lines:
            weeks = re.findall(r"[Ww]eek (\d+)", lines[0])
            found[p] = "week " + weeks[0] if weeks else "week ?"
    print("→", found)

## 3. Cut to fit and send once

The simplest way: keep only the first 1,500 tokens and ask once

In [ ]:
first_part = tok.decode(tok.encode(DOC, add_special_tokens=False).ids[:MAX])
report(call(f"{QUERY}\n\nContext:\n{first_part}"))

## 4. The textbook's `recursive_summarize`

1. **Base case:** if the context fits the budget → call the model once
2. **Recursive case:** split by structure into chunks of at most half the budget → call itself on each chunk with a sub-question
3. **Aggregate:** join all the sub-results and ask the original question again

"Split by structure" for this document: by `## Week` headings · if a single heading is still too big, by paragraphs

In [ ]:
KEEP_HEADING = False

def split_by_structure(text, max_tokens):
    units, head = re.split(r"\n(?=## )", text), ""
    if len(units) == 1:              # a single section that is still too big → split it by paragraphs
        head, body = text.split("\n", 1)
        units = body.split("\n\n")

    # ✏️ TODO 1 (▶ video 3:31): if adding this unit would go over the budget, close the current chunk and start a new one with this unit

    chunks, cur = [], ""
    for u in units:
        if cur and ___:
            chunks.append(cur)
            cur = ___
        else:
            cur = cur + "\n\n" + u if cur else u
    chunks.append(cur)
    if head and KEEP_HEADING:
        chunks = [head + "\n" + c for c in chunks]
    return chunks

In [ ]:
# ✏️ TODO 2 (▶ video 3:50): base case: if the context fits the budget, call the model once
def recursive_summarize(context, query, max_tokens, depth=0):
    n = count_tokens(context)
    print("    " * depth + f"depth {depth}: {n:,} tokens")
    if ___:
        return call(f"{query}\n\nContext:\n{context}", depth)

    chunks = split_by_structure(context, max_tokens // 2)
    sub_results = [recursive_summarize(c, f"Summarize this section relevant to: {query}", max_tokens, depth + 1)
                   for c in chunks]

    # ✏️ TODO 3 (▶ video 4:16): aggregate: join all the sub-results and ask the original question again

    combined = ___
    return call(f"Given these partial summaries, answer: {query}\n\nSummaries:\n{combined}", depth)

In [ ]:
CALLS.clear()
report(recursive_summarize(DOC, QUERY, MAX))

## 5. The heading lost in the split

Week 27 was split by paragraphs at depth 2 — the chunk with the P04 decision **has no `## Week 27` line** in it at all
so the model reading that chunk has no way to know which week it is → fix it in the harness: carry the heading into every chunk

In [ ]:
print(split_by_structure(weeks[26], MAX // 2)[1][:200])

In [ ]:
KEEP_HEADING = True
print(split_by_structure(weeks[26], MAX // 2)[1][:200])

In [ ]:
CALLS.clear()
report(recursive_summarize(DOC, QUERY, MAX))

## 6. The cost of recursion

No call sees the whole document — but the model is called many times
Compare with sending the whole document once (8-thousand-odd tokens **fit Gemma's real window easily** — the 1,500 budget is the lab's assumption)

In [ ]:
tree = list(CALLS)
print(f"recursive: {len(tree)} calls · largest call {max(c[1] for c in tree):,} tokens · "
      f"total input {sum(c[1] for c in tree):,} tokens · deepest level {max(c[0] for c in tree)}")
CALLS.clear()
report(call(f"{QUERY}\n\nContext:\n{DOC}"))
print(f"one call: {CALLS[0][1]:,} tokens")

In [ ]:
cost = sum(c[1] for c in tree + CALLS) / 1e6 * 0.07 + sum(c[2] for c in tree + CALLS) / 1e6 * 0.34
print(f"the recursive run + the one-call run ≈ ${cost:.4f}")

## Summary

| What we did | What we saw in the lab |
|---|---|
| Cut to fit, send once | Only the start is seen → finds only P09 |
| `recursive_summarize` | A tree two levels deep (week 27 branches again) · all 3 papers found, but P04 has no week |
| Carry the heading into every chunk | All 3 papers with weeks 9 · 27 · 44 · not fooled by the P05 decoy |
| Cost | 15 calls · the largest call within budget · slightly more tokens in total than one call |

**Exercises**
1. Lower `MAX` to 800 — does the tree get deeper, how many calls does it take, and is the answer still complete?
2. The textbook says this pattern also works for finding a needle in a haystack (recursive search): change `QUERY` to *"Who suggested dropping P05?"* and run it again

**Next — Lab 4:** Lab 2 again, with a framework — LangChain's `SummarizationMiddleware`